In [26]:
# --------------------------------------------------
# Project Root
# --------------------------------------------------

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\IITG\Projects\audio_factor_disentanglement_v2


In [27]:
# --------------------------------------------------
# Imports
# --------------------------------------------------

import pandas as pd
import numpy as np

from pprint import pprint

In [28]:
PROJECT_GOAL = {

    "preserve_content": True,

    "preserve_speaker": True,

    "preserve_excitation": True,

    "swap_environment": True,

    "swap_microphone": True,

    "preserve_fidelity": True,

    "waveform_reconstruction": True
}

PROJECT_GOAL

{'preserve_content': True,
 'preserve_speaker': True,
 'preserve_excitation': True,
 'swap_environment': True,
 'swap_microphone': True,
 'preserve_fidelity': True,
 'waveform_reconstruction': True}

In [29]:
feature_roles = {

    "logmel": {

        "content": 5,
        "speaker": 4,
        "environment": 2,
        "fidelity": 3
    },

    "mr_mag_256": {

        "content": 4,
        "speaker": 5,
        "environment": 2,
        "fidelity": 3
    },

    "mr_mag_512": {

        "content": 4,
        "speaker": 5,
        "environment": 3,
        "fidelity": 5
    },

    "magnitude": {

        "content": 3,
        "speaker": 3,
        "environment": 5,
        "fidelity": 4
    },

    "mr_mag_1024": {

        "content": 3,
        "speaker": 3,
        "environment": 5,
        "fidelity": 5
    },

    "if": {

        "content": 2,
        "speaker": 1,
        "environment": 5,
        "fidelity": 2
    },

    "modgd": {

        "content": 4,
        "speaker": 2,
        "environment": 1,
        "fidelity": 4
    },

    "phase_sin": {

        "content": 1,
        "speaker": 1,
        "environment": 1,
        "fidelity": 5
    },

    "phase_cos": {

        "content": 1,
        "speaker": 1,
        "environment": 1,
        "fidelity": 5
    }
}

pd.DataFrame(feature_roles).T

,content,speaker,environment,fidelity
logmel,5,4,2,3
mr_mag_256,4,5,2,3
mr_mag_512,4,5,3,5
magnitude,3,3,5,4
mr_mag_1024,3,3,5,5
if,2,1,5,2
modgd,4,2,1,4
phase_sin,1,1,1,5
phase_cos,1,1,1,5


In [30]:
latent_groups = {

    "content": [

        "logmel",
        "mr_mag_256",
        "if"
    ],

    "speaker": [

        "mr_mag_512",
        "mr_mag_256",
        "logmel"
    ],

    "environment": [

        "magnitude",
        "mr_mag_1024",
        "if"
    ],

    "excitation": [

        "modgd"
    ],

    "fidelity": [

        "phase_sin",
        "phase_cos",
        "mr_mag_512",
        "mr_mag_1024",
        "magnitude",
        "modgd"
    ]
}

pprint(latent_groups)

{'content': ['logmel', 'mr_mag_256', 'if'],
 'environment': ['magnitude', 'mr_mag_1024', 'if'],
 'excitation': ['modgd'],
 'fidelity': ['phase_sin',
              'phase_cos',
              'mr_mag_512',
              'mr_mag_1024',
              'magnitude',
              'modgd'],
 'speaker': ['mr_mag_512', 'mr_mag_256', 'logmel']}


In [31]:
latent_dims = {

    "content": 64,

    "speaker": 64,

    "environment": 96,

    "excitation": 32,

    "fidelity": 128
}

latent_dims

{'content': 64,
 'speaker': 64,
 'environment': 96,
 'excitation': 32,
 'fidelity': 128}

In [32]:
print(
    "Total Latent Dim =",
    sum(latent_dims.values())
)

Total Latent Dim = 384


In [33]:
pd.DataFrame({

    "group": latent_dims.keys(),

    "latent_dim": latent_dims.values()
})

,group,latent_dim
0,content,64
1,speaker,64
2,environment,96
3,excitation,32
4,fidelity,128


In [34]:
encoder_blueprints = {

    "content": {

        "features": [

            "logmel",
            "mr_mag_256",
            "if"
        ],

        "blocks": [

            "ConvStem",

            "Conformer x4",

            "TemporalAttention",

            "VariationalProjection"
        ],

        "latent_dim": 64
    },

    "speaker": {

        "features": [

            "mr_mag_512",
            "mr_mag_256",
            "logmel"
        ],

        "blocks": [

            "ConvStem",

            "Conformer x6",

            "AttentiveStatsPooling",

            "TransformerPooling",

            "VariationalProjection"
        ],

        "latent_dim": 64
    },

    "environment": {

        "features": [

            "magnitude",
            "mr_mag_1024",
            "if"
        ],

        "blocks": [

            "ConvStem",

            "DilatedTCN",

            "Transformer",

            "GlobalContextPooling",

            "VariationalProjection"
        ],

        "latent_dim": 96
    },

    "excitation": {

        "features": [

            "modgd"
        ],

        "blocks": [

            "ConvStem",

            "Conformer x3",

            "TemporalAttention",

            "VariationalProjection"
        ],

        "latent_dim": 32
    },

    "fidelity": {

        "features": [

            "phase_sin",
            "phase_cos",

            "mr_mag_512",
            "mr_mag_1024",

            "magnitude",
            "modgd"
        ],

        "blocks": [

            "MultiResolutionConv",

            "Transformer x6",

            "CrossScaleAttention",

            "VariationalProjection"
        ],

        "latent_dim": 128
    }
}

encoder_blueprints

{'content': {'features': ['logmel', 'mr_mag_256', 'if'],
  'blocks': ['ConvStem',
   'Conformer x4',
   'TemporalAttention',
   'VariationalProjection'],
  'latent_dim': 64},
 'speaker': {'features': ['mr_mag_512', 'mr_mag_256', 'logmel'],
  'blocks': ['ConvStem',
   'Conformer x6',
   'AttentiveStatsPooling',
   'TransformerPooling',
   'VariationalProjection'],
  'latent_dim': 64},
 'environment': {'features': ['magnitude', 'mr_mag_1024', 'if'],
  'blocks': ['ConvStem',
   'DilatedTCN',
   'Transformer',
   'GlobalContextPooling',
   'VariationalProjection'],
  'latent_dim': 96},
 'excitation': {'features': ['modgd'],
  'blocks': ['ConvStem',
   'Conformer x3',
   'TemporalAttention',
   'VariationalProjection'],
  'latent_dim': 32},
 'fidelity': {'features': ['phase_sin',
   'phase_cos',
   'mr_mag_512',
   'mr_mag_1024',
   'magnitude',
   'modgd'],
  'blocks': ['MultiResolutionConv',
   'Transformer x6',
   'CrossScaleAttention',
   'VariationalProjection'],
  'latent_dim': 128}}

In [35]:
decoder_design = {

    "identity_backbone": {

        "inputs": [

            "z_content",
            "z_speaker"
        ],

        "block": [

            "Concat",

            "IdentityTransformer",

            "IdentityTokens"
        ]
    },

    "environment_conditioning": {

        "latent":

            "z_environment",

        "mechanism":

            "FiLM"
    },

    "excitation_conditioning": {

        "latent":

            "z_excitation",

        "mechanism":

            "CrossAttention"
    },

    "fidelity_conditioning": {

        "latent":

            "z_fidelity",

        "mechanism":

            "MultiHeadCrossAttention"
    }
}

decoder_design

{'identity_backbone': {'inputs': ['z_content', 'z_speaker'],
  'block': ['Concat', 'IdentityTransformer', 'IdentityTokens']},
 'environment_conditioning': {'latent': 'z_environment', 'mechanism': 'FiLM'},
 'excitation_conditioning': {'latent': 'z_excitation',
  'mechanism': 'CrossAttention'},
 'fidelity_conditioning': {'latent': 'z_fidelity',
  'mechanism': 'MultiHeadCrossAttention'}}

In [36]:
reconstruction_heads = {

    "content_head": [

        "logmel",
        "mr_mag_256"
    ],

    "speaker_head": [

        "mr_mag_512"
    ],

    "environment_head": [

        "magnitude",
        "mr_mag_1024",
        "if"
    ],

    "excitation_head": [

        "modgd"
    ],

    "fidelity_head": [

        "phase_sin",
        "phase_cos"
    ]
}

reconstruction_heads

{'content_head': ['logmel', 'mr_mag_256'],
 'speaker_head': ['mr_mag_512'],
 'environment_head': ['magnitude', 'mr_mag_1024', 'if'],
 'excitation_head': ['modgd'],
 'fidelity_head': ['phase_sin', 'phase_cos']}

In [37]:
betas = {

    "content": 1,

    "speaker": 2,

    "environment": 16,

    "excitation": 4,

    "fidelity": 1
}

pd.DataFrame({

    "latent": betas.keys(),

    "beta": betas.values()
})

,latent,beta
0,content,1
1,speaker,2
2,environment,16
3,excitation,4
4,fidelity,1


In [38]:
loss_graph = {

    "reconstruction": [

        "L1",

        "FeatureMSE",

        "MultiResolutionSTFTLoss"
    ],

    "variational": [

        "KL_content",

        "KL_speaker",

        "KL_environment",

        "KL_excitation",

        "KL_fidelity"
    ],

    "disentanglement": [

        "FactorVAE",

        "TotalCorrelation",

        "OrthogonalityLoss"
    ],

    "representation": [

        "InfoNCE"
    ],

    "transfer": [

        "EnvironmentConsistencyLoss"
    ]
}

loss_graph

{'reconstruction': ['L1', 'FeatureMSE', 'MultiResolutionSTFTLoss'],
 'variational': ['KL_content',
  'KL_speaker',
  'KL_environment',
  'KL_excitation',
  'KL_fidelity'],
 'disentanglement': ['FactorVAE', 'TotalCorrelation', 'OrthogonalityLoss'],
 'representation': ['InfoNCE'],
 'transfer': ['EnvironmentConsistencyLoss']}

In [39]:
orthogonality_pairs = [

    ("content","speaker"),

    ("content","environment"),

    ("speaker","environment"),

    ("speaker","excitation"),

    ("environment","excitation")
]

pd.DataFrame(
    orthogonality_pairs,
    columns=[
        "latent_a",
        "latent_b"
    ]
)

,latent_a,latent_b
0,content,speaker
1,content,environment
2,speaker,environment
3,speaker,excitation
4,environment,excitation


In [40]:
swap_experiment = {

    "content":
        "s2",

    "speaker":
        "s2",

    "environment":
        "s1",

    "excitation":
        "s2",

    "fidelity":
        "s2"
}

swap_experiment

{'content': 's2',
 'speaker': 's2',
 'environment': 's1',
 'excitation': 's2',
 'fidelity': 's2'}

In [41]:
environment_consistency = {

    "reference":

        "z_environment(s1_clean)",

    "generated":

        "EnvEncoder(decoded)",

    "loss":

        "MSE"
}

environment_consistency

{'reference': 'z_environment(s1_clean)',
 'generated': 'EnvEncoder(decoded)',
 'loss': 'MSE'}

In [42]:
pipeline = """

s1_clean
    ↓
EnvironmentEncoder
    ↓
z_environment_clean

------------------------------------------------

s2_noisy
    ↓
ContentEncoder
SpeakerEncoder
ExcitationEncoder
FidelityEncoder

------------------------------------------------

Decoder(

    z_content_s2,

    z_speaker_s2,

    z_environment_clean,

    z_excitation_s2,

    z_fidelity_s2
)

↓

Feature Reconstruction

↓

Mel Projection

↓

BigVGAN

↓

Waveform

↓

Fragment Stitching

↓

Final Audio

"""

print(pipeline)



s1_clean
    ↓
EnvironmentEncoder
    ↓
z_environment_clean

------------------------------------------------

s2_noisy
    ↓
ContentEncoder
SpeakerEncoder
ExcitationEncoder
FidelityEncoder

------------------------------------------------

Decoder(

    z_content_s2,

    z_speaker_s2,

    z_environment_clean,

    z_excitation_s2,

    z_fidelity_s2
)

↓

Feature Reconstruction

↓

Mel Projection

↓

BigVGAN

↓

Waveform

↓

Fragment Stitching

↓

Final Audio




In [43]:
FINAL_ARCHITECTURE = """

Factorized Environment Transfer VAE

Latents:

z_content      = preserve

z_speaker      = preserve

z_environment  = transfer

z_excitation   = preserve

z_fidelity     = preserve


Transfer Experiment:

content      ← s2
speaker      ← s2
environment  ← s1
excitation   ← s2
fidelity     ← s2


Expected Output:

same words

same speaker

same energy

clean environment

clean microphone

high fidelity waveform

"""

print(FINAL_ARCHITECTURE)



Factorized Environment Transfer VAE

Latents:

z_content      = preserve

z_speaker      = preserve

z_environment  = transfer

z_excitation   = preserve

z_fidelity     = preserve


Transfer Experiment:

content      ← s2
speaker      ← s2
environment  ← s1
excitation   ← s2
fidelity     ← s2


Expected Output:

same words

same speaker

same energy

clean environment

clean microphone

high fidelity waveform


